This notebook defines the model that will be used to predict the performance of a new store for the swap engine.

The idea is to train a model that predicts:
- capture rate
- sales per sqm
- dwell time

Given inputs:
- gla of block
- gla category of block
- bl1_label
- avg & median window flow of store
- neighborhood synergy metric from synergy graph
- mall id
- mall total gla
- number of stores in mall
- mall category distribution (eg % of f&b, % of fashion, etc)

# Imports

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_score
from sklearn.preprocessing import LabelEncoder

# Data Loading

In [ ]:
import constants.constants as cst
import constants.paths as pth

In [ ]:
# Dim Tables
dim_blocks = pd.read_csv(pth.INTERMEDIATE_DIM_BLOCKS, **cst.CSV_PARAMS)
dim_malls = pd.read_csv(pth.INTERMEDIATE_DIM_MALLS, **cst.CSV_PARAMS)

# Fact Tables
fact_stores = pd.read_csv(pth.INTERMEDIATE_FACT_STORES, **cst.CSV_PARAMS)
fact_malls = pd.read_csv(pth.INTERMEDIATE_FACT_MALLS, **cst.CSV_PARAMS)
fact_sri_scores = pd.read_csv(pth.INTERMEDIATE_FACT_SRI_SCORES, **cst.CSV_PARAMS)

# Store financials table
store_financials = pd.read_csv(pth.INTERMEDIATE_STORE_FINANCIALS, **cst.CSV_PARAMS)

# Cross visits table
cross_visits = pd.read_csv(pth.INTERMEDIATE_CROSS_VISITS, **cst.CSV_PARAMS)

# Neighborhood affinity table
affinity = pd.read_csv(pth.ENRICHED_CATEGORY_AFFINITIES, **cst.CSV_PARAMS)

# Data Preparation

In [ ]:
import constants.column_names as col

Compute beforehand:

In [ ]:
# For stores that span multiple blocks, we take the sum of the people window flow
# over all blocks
store_daily = (
    fact_stores.groupby([col.STORE_CODE, col.DATE])
    .agg(
        **{
            col.PEOPLE_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "sum"),
            col.PEOPLE_IN: (col.PEOPLE_IN, "sum"),
            col.STORE_AVG_DWELL_TIME: (col.STORE_AVG_DWELL_TIME, "mean"),
        }
    )
    .reset_index()
)

store_total = store_daily.groupby(col.STORE_CODE).agg(
    **{
        col.MODEL_STORE_AVG_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "mean"),
        col.MODEL_STORE_MEDIAN_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "median"),
        col.MODEL_STORE_TOTAL_PEOPLE_IN: (col.PEOPLE_IN, "sum"),
        col.MODEL_STORE_TOTAL_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "sum"),
        col.MODEL_STORE_DAYS_RECORDED: (col.DATE, "nunique"),
        col.MODEL_STORE_AVG_DWELL_TIME: (col.STORE_AVG_DWELL_TIME, "mean"),
    }
)

# Feature Engineering

In [ ]:
def engineer_store_features(
    store_code: int,
    dim_blocks: pd.DataFrame,
    store_total: pd.DataFrame,
    cross_visits: pd.DataFrame,
    affinity_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Engineer features for a given store.

    Args:
        store_code (int): The store code.
        dim_blocks (pd.DataFrame): Dimension table for blocks.
        store_total (pd.DataFrame): Aggregated store data over the entire period.
        cross_visits (pd.DataFrame): Cross visits data between stores.
        affinity_matrix (pd.DataFrame): Affinity matrix for categories.

    Returns:
        dict: A dictionary of engineered features for the store.
    """
    features = {}

    # There are duplicate store codes in dim_blocks, correspond to stores that span over
    # multiple blocks. The gla corresponds to the sum of the gla of all blocks, so we
    # can take any of the rows for other store attributes.
    store_info = dim_blocks[dim_blocks[col.STORE_CODE] == store_code].iloc[0]
    mall_id = store_info[col.MALL_ID]
    store_category = store_info[col.CAT_HIGH]

    # Get neighboring stores based on cross visits
    store_cross = cross_visits[
        (cross_visits[col.STORE_CODE_1] == store_code)
        | (cross_visits[col.STORE_CODE_2] == store_code)
    ]

    neighbor_codes = set(store_cross[col.STORE_CODE_1]) | set(
        store_cross[col.STORE_CODE_2]
    )
    neighbor_codes.discard(store_code)

    # Get category distribution of neighboring stores
    neighbor_categories = dim_blocks[dim_blocks[col.STORE_CODE].isin(neighbor_codes)][
        col.CAT_HIGH
    ].value_counts()

    # Compute synergy score: sum of (affinity × neighbor_count) for each neighbor
    # category
    synergy_score = 0
    for neighbor_cat, count in neighbor_categories.items():
        affinity_score = affinity_matrix.loc[
            (affinity_matrix[col.CATEGORY_A] == store_category)
            & (affinity_matrix[col.CATEGORY_B] == neighbor_cat)
        ][col.AFFINITY].values
        if len(affinity_score) > 0 and pd.notna(affinity_score[0]):
            synergy_score += affinity_score[0] * count

    mall_stores = dim_blocks[dim_blocks[col.MALL_ID] == mall_id].drop_duplicates(
        subset=[col.STORE_CODE]
    )

    #### Features ####
    # Mall ID (categorical)
    features[col.MALL_ID] = mall_id

    # Intrinsic store features
    features[col.MODEL_STORE_GLA] = store_info[col.GLA]
    features[col.MODEL_STORE_GLA_CAT] = store_info[col.GLA_CAT]
    features[col.MODEL_STORE_CATEGORY] = store_category

    # Location features
    features[col.MODEL_STORE_AVG_WINDOW_FLOW] = store_total.loc[
        store_code, col.MODEL_STORE_AVG_WINDOW_FLOW
    ]
    features[col.MODEL_STORE_MEDIAN_WINDOW_FLOW] = store_total.loc[
        store_code, col.MODEL_STORE_MEDIAN_WINDOW_FLOW
    ]

    # Neighborhood synergy feature
    features[col.MODEL_STORE_NEIGHBORHOOD_SYNERGY] = synergy_score
    features[col.MODEL_STORE_NB_NEIGHBORS] = len(neighbor_codes)

    # Mall features
    features[col.MODEL_MALL_TOTAL_GLA] = mall_stores[col.GLA].sum()
    features[col.MODEL_MALL_STORE_COUNT] = len(mall_stores)
    features[col.MODEL_MALL_CATEGORY_SHARE] = (
        mall_stores[mall_stores[col.CAT_HIGH] == store_category][col.GLA].sum()
        / features[col.MODEL_MALL_TOTAL_GLA]
    )

    return features

In [ ]:
engineer_store_features(
    42,
    dim_blocks,
    store_total,
    cross_visits,
    affinity,
)

# Training Dataset Preparation

In [ ]:
def build_training_dateset(
    dim_blocks: pd.DataFrame,
    store_total: pd.DataFrame,
    cross_visits: pd.DataFrame,
    affinity_matrix: pd.DataFrame,
    store_financials: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, dict, list]:
    """Build the training dataset by engineering features for all stores.

    Args:
        dim_blocks (pd.DataFrame): Dimension table for blocks.
        store_total (pd.DataFrame): Aggregated store data.
        cross_visits (pd.DataFrame): Cross visits data between stores.
        affinity_matrix (pd.DataFrame): Affinity matrix for categories.
        store_financials (pd.DataFrame): Store financials data (for target variable).

    Returns:
        pd.DataFrame: The training dataset with engineered features for all stores.
        pd.DataFrame: The target variables (normalized by mall mean).
        pd.Series: Mall IDs for each store (for leave-one-mall-out CV).
        dict: Mall means for each target (to convert predictions back to absolute).
        list: Store codes in the same order as features/targets (for index alignment).
    """
    # There are some stores in store_total that are not in dim_blocks, which we skip
    # as we cannot engineer features for them (like category, mall, etc.)
    valid_store_codes = set(dim_blocks[col.STORE_CODE]) & set(store_total.index)
    store_total = store_total.loc[list(valid_store_codes)]

    # Merge financials and GLA
    store_total_merged = pd.merge(
        store_total,
        store_financials,
        on=col.STORE_CODE,
        how="left",
        validate="1:1",
    )

    store_total_merged = pd.merge(
        store_total_merged,
        dim_blocks[[col.STORE_CODE, col.GLA]].drop_duplicates(subset=[col.STORE_CODE]),
        on=col.STORE_CODE,
        how="left",
        validate="1:1",
    )

    store_total_merged = store_total_merged.set_index(col.STORE_CODE)

    # Engineer features for all stores
    feature_list = []
    store_codes_order = []  # Track the order of stores
    for store_code in store_total.index:
        features = engineer_store_features(
            store_code,
            dim_blocks,
            store_total,
            cross_visits,
            affinity_matrix,
        )
        feature_list.append(pd.Series(features, name=store_code))
        store_codes_order.append(store_code)

    features_df = pd.DataFrame(feature_list)
    features_df.index.name = col.STORE_CODE

    # Compute raw targets aligned with features index
    targets_raw = pd.DataFrame(index=features_df.index)
    targets_raw[col.TARGET_CAPTURE_RATE] = (
        store_total_merged.loc[features_df.index, col.MODEL_STORE_TOTAL_PEOPLE_IN]
        / store_total_merged.loc[features_df.index, col.MODEL_STORE_TOTAL_WINDOW_FLOW]
    )
    targets_raw[col.TARGET_SALES_PER_SQM] = (
        store_total_merged.loc[features_df.index, col.SALES_R12M]
        / store_total_merged.loc[features_df.index, col.GLA]
    )
    targets_raw[col.TARGET_DWELL_TIME] = store_total_merged.loc[
        features_df.index, col.MODEL_STORE_AVG_DWELL_TIME
    ]

    # Replace inf values with NaN to avoid polluting mall means
    import numpy as np

    targets_raw = targets_raw.replace([np.inf, -np.inf], np.nan)

    # Get mall_ids for normalization and CV
    store_to_mall = dim_blocks.drop_duplicates(subset=[col.STORE_CODE]).set_index(
        col.STORE_CODE
    )[col.MALL_ID]
    mall_ids = store_to_mall.loc[features_df.index]

    # Compute mall means for each target (excluding inf/nan values)
    targets_raw[col.MALL_ID] = mall_ids.values
    mall_means = {}
    for target_col in [
        col.TARGET_CAPTURE_RATE,
        col.TARGET_SALES_PER_SQM,
        col.TARGET_DWELL_TIME,
    ]:
        # Use only finite values for computing mall means
        mall_means[target_col] = (
            targets_raw.groupby(col.MALL_ID)[target_col].mean().to_dict()
        )

    # Normalize targets by mall mean (1.0 = mall average)
    targets = pd.DataFrame(index=features_df.index)
    for target_col in [
        col.TARGET_CAPTURE_RATE,
        col.TARGET_SALES_PER_SQM,
        col.TARGET_DWELL_TIME,
    ]:
        mall_mean_series = mall_ids.map(mall_means[target_col])
        targets[target_col] = targets_raw[target_col] / mall_mean_series

    # Reset indexes for alignment
    mall_ids = mall_ids.reset_index(drop=True)
    features_df = features_df.reset_index(drop=True)
    targets = targets.reset_index(drop=True)

    return features_df, targets, mall_ids, mall_means, store_codes_order

In [ ]:
X, y, mall_ids, mall_means, store_codes = build_training_dateset(
    dim_blocks,
    store_total,
    cross_visits,
    affinity,
    store_financials,
)

# Show what the normalized targets look like
print("Normalized targets (1.0 = mall average):")
print(y.describe())

# Model Training

In [ ]:
def train_performance_model(X, y, target_col, mall_ids, cv_strategy="kfold"):
    """Train a performance prediction model.

    Args:
        X: Feature DataFrame
        y: Target DataFrame (normalized by mall mean, so 1.0 = average)
        target_col: Which target to predict
        mall_ids: Series with mall_id for each row (aligned with X and y)
        cv_strategy: "kfold" (default) or "leave_mall_out"

    Returns:
        Trained model, encoders, CV scores
    """
    # Mask rows where target is NaN
    mask_target = y[target_col].notna()
    X_filtered = X.loc[mask_target].copy()
    y_filtered = y.loc[mask_target]
    groups = mall_ids.loc[mask_target]

    # Encode categorical features
    cat_cols = X_filtered.select_dtypes(include=["object"]).columns.tolist()
    # mall_id is int, need to add it to categorical encoding
    if col.MALL_ID in X_filtered.columns:
        cat_cols.append(col.MALL_ID)

    encoders = {}
    for column in cat_cols:
        le = LabelEncoder()
        X_filtered[column] = le.fit_transform(X_filtered[column].astype(str))
        encoders[column] = le

    # Prepare target
    y_target = y_filtered[target_col]

    # Model
    model = RandomForestRegressor(
        n_estimators=1000, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1
    )

    # Cross-validation
    if cv_strategy == "kfold":
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_filtered, y_target, cv=cv, scoring="r2")
    else:
        # Leave-one-mall-out (note: mall_id feature won't generalize to unseen malls)
        logo = LeaveOneGroupOut()
        cv_scores = cross_val_score(
            model, X_filtered, y_target, cv=logo, groups=groups, scoring="r2"
        )

    print(f"Target: {target_col}")
    print(f"CV strategy: {cv_strategy}")
    print(f"CV R² scores: {cv_scores}")
    print(f"Mean R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

    # Fit final model on all data
    model.fit(X_filtered, y_target)

    return model, encoders, cv_scores

In [ ]:
# Train models for both sales_per_sqm and dwell_time
print("Training sales_per_sqm model...")
model_sales, encoders_sales, cv_sales = train_performance_model(
    X, y, col.TARGET_SALES_PER_SQM, mall_ids, cv_strategy="kfold"
)

print("\nTraining dwell_time model...")
model_dwell, encoders_dwell, cv_dwell = train_performance_model(
    X, y, col.TARGET_DWELL_TIME, mall_ids, cv_strategy="kfold"
)

# Store models in a dict for predict_swap_impact
models = {
    "sales_per_sqm": model_sales,
    "dwell_time": model_dwell,
}

# Use the same encoders (they should be identical since same features)
encoders = encoders_sales

# Swap Impact Prediction

In [ ]:
def compute_synergy_score(
    store_category: str,
    neighbor_categories: pd.Series,
    affinity_matrix: pd.DataFrame,
) -> float:
    """Compute synergy score for a store given its neighbors' categories.

    Args:
        store_category: The category of the store.
        neighbor_categories: Series with category counts of neighboring stores.
        affinity_matrix: Affinity matrix for categories.

    Returns:
        Synergy score (sum of affinity × neighbor_count for each neighbor category).
    """
    synergy_score = 0.0
    for neighbor_cat, count in neighbor_categories.items():
        affinity_row = affinity_matrix.loc[
            (affinity_matrix[col.CATEGORY_A] == store_category)
            & (affinity_matrix[col.CATEGORY_B] == neighbor_cat)
        ]
        if len(affinity_row) > 0:
            affinity_val = affinity_row[col.AFFINITY].values[0]
            if pd.notna(affinity_val):
                synergy_score += affinity_val * count
    return synergy_score


def get_store_neighbors(store_code: int, cross_visits: pd.DataFrame) -> set:
    """Get all neighbor store codes for a given store."""
    store_cross = cross_visits[
        (cross_visits[col.STORE_CODE_1] == store_code)
        | (cross_visits[col.STORE_CODE_2] == store_code)
    ]
    neighbor_codes = set(store_cross[col.STORE_CODE_1]) | set(
        store_cross[col.STORE_CODE_2]
    )
    neighbor_codes.discard(store_code)
    return neighbor_codes


def encode_features_for_prediction(
    features: dict,
    encoders: dict,
) -> pd.DataFrame:
    """Encode a single store's features for model prediction.

    Args:
        features: Dictionary of features for a store.
        encoders: Dictionary of LabelEncoders for categorical columns.

    Returns:
        DataFrame with encoded features ready for prediction.
    """
    features_df = pd.DataFrame([features])
    for column, encoder in encoders.items():
        if column in features_df.columns:
            val = str(features_df[column].iloc[0])
            if val in encoder.classes_:
                features_df[column] = encoder.transform([val])[0]
            else:
                features_df[column] = 0
    return features_df


def compute_category_sri_averages(
    fact_sri_scores: pd.DataFrame,
    dim_blocks: pd.DataFrame,
) -> dict:
    """Compute average SRI score for each category.

    Args:
        fact_sri_scores: DataFrame with store_code and sri_score.
        dim_blocks: Dimension table for blocks (to get category).

    Returns:
        Dictionary mapping category to average SRI score.
    """
    sri_with_cat = pd.merge(
        fact_sri_scores,
        dim_blocks[[col.STORE_CODE, col.CAT_HIGH]].drop_duplicates(),
        on=col.STORE_CODE,
        how="left",
    )
    return sri_with_cat.groupby(col.CAT_HIGH)[col.SRI_SCORE].mean().to_dict()


def compute_gla_weighted_sri(
    store_sri: pd.Series,
    store_gla: pd.Series,
) -> float:
    """Compute GLA-weighted average SRI score.

    Args:
        store_sri: Series with SRI scores indexed by store_code.
        store_gla: Series with GLA indexed by store_code.

    Returns:
        GLA-weighted average SRI (larger stores contribute more).
    """
    # Align indices - only include stores that have both SRI and GLA
    common_stores = store_sri.index.intersection(store_gla.index)
    sri_aligned = store_sri.loc[common_stores]
    gla_aligned = store_gla.loc[common_stores]

    # Drop NaN values
    valid_mask = sri_aligned.notna() & gla_aligned.notna()
    sri_valid = sri_aligned[valid_mask]
    gla_valid = gla_aligned[valid_mask]

    if gla_valid.sum() == 0:
        return sri_valid.mean() if len(sri_valid) > 0 else 0.0

    return (sri_valid * gla_valid).sum() / gla_valid.sum()


def compute_mall_composite_score(
    avg_sales: float,
    avg_dwell: float,
    avg_sri: float,
    weights: dict = None,
) -> float:
    """Compute weighted composite mall score.

    Args:
        avg_sales: Average sales per sqm (normalized, 1.0 = baseline).
        avg_dwell: Average dwell time (normalized, 1.0 = baseline).
        avg_sri: Average SRI score (raw score, typically 10-50).
        weights: Dictionary with weights for each metric. Default: equal weights.

    Returns:
        Composite score (higher is better).
    """
    if weights is None:
        weights = {"sales": 0.4, "dwell": 0.3, "sri": 0.3}

    # Normalize SRI to similar scale (assuming typical range 10-50, normalize to ~1.0)
    sri_normalized = avg_sri / 30.0  # 30 as a reasonable midpoint

    score = (
        weights["sales"] * avg_sales
        + weights["dwell"] * avg_dwell
        + weights["sri"] * sri_normalized
    )
    return score


def predict_swap_impact(
    store_to_swap: int,
    new_category: str,
    models: dict,
    encoders: dict,
    dim_blocks: pd.DataFrame,
    store_total: pd.DataFrame,
    cross_visits: pd.DataFrame,
    affinity_matrix: pd.DataFrame,
    current_store_performance: pd.DataFrame,
    current_store_sri: pd.DataFrame,
    category_sri_avg: dict,
    weights: dict = None,
) -> dict:
    """Predict the impact of swapping a store to a new category.

    Computes mall-level composite score using:
    - Average sales per sqm
    - Average dwell time
    - GLA-weighted average SRI score (larger stores have more impact)

    Args:
        store_to_swap: Store code of the store to swap.
        new_category: The new bl1_label category to swap to.
        models: Dict with trained models for 'sales_per_sqm' and 'dwell_time'.
        encoders: Dictionary of LabelEncoders for categorical features.
        dim_blocks: Dimension table for blocks.
        store_total: Aggregated store data.
        cross_visits: Cross visits data.
        affinity_matrix: Affinity matrix for categories.
        current_store_performance: DataFrame with current relative performance.
        current_store_sri: Series with SRI scores indexed by store_code.
        category_sri_avg: Dict mapping category to average SRI.
        weights: Dict with weights for composite score (sales, dwell, sri).

    Returns:
        Dictionary with current and predicted mall metrics and composite scores.
    """
    # Get store info
    store_info = dim_blocks[dim_blocks[col.STORE_CODE] == store_to_swap].iloc[0]
    mall_id = store_info[col.MALL_ID]
    old_category = store_info[col.CAT_HIGH]
    store_gla = store_info[col.GLA]

    # Get all stores in the mall with their GLA
    mall_stores = dim_blocks[dim_blocks[col.MALL_ID] == mall_id].drop_duplicates(
        subset=[col.STORE_CODE]
    )
    mall_store_codes = set(mall_stores[col.STORE_CODE])
    mall_store_gla = mall_stores.set_index(col.STORE_CODE)[col.GLA]

    # Get neighbors of the store to swap
    neighbors = get_store_neighbors(store_to_swap, cross_visits)

    # ============ BUILD NEW STORE FEATURES ============
    new_store_features = engineer_store_features(
        store_to_swap, dim_blocks, store_total, cross_visits, affinity_matrix
    )
    new_store_features[col.MODEL_STORE_CATEGORY] = new_category

    # Recalculate synergy with new category
    neighbor_categories = dim_blocks[dim_blocks[col.STORE_CODE].isin(neighbors)][
        col.CAT_HIGH
    ].value_counts()
    new_store_features[col.MODEL_STORE_NEIGHBORHOOD_SYNERGY] = compute_synergy_score(
        new_category, neighbor_categories, affinity_matrix
    )

    # Recalculate category share in mall
    current_new_cat_gla = mall_stores[mall_stores[col.CAT_HIGH] == new_category][
        col.GLA
    ].sum()
    mall_total_gla = new_store_features[col.MODEL_MALL_TOTAL_GLA]
    new_store_features[col.MODEL_MALL_CATEGORY_SHARE] = (
        current_new_cat_gla + store_gla
    ) / mall_total_gla

    # ============ PREDICT NEW STORE PERFORMANCE ============
    new_store_encoded = encode_features_for_prediction(new_store_features, encoders)

    pred_sales = models["sales_per_sqm"].predict(new_store_encoded)[0]
    pred_dwell = models["dwell_time"].predict(new_store_encoded)[0]
    pred_sri = category_sri_avg.get(new_category, 25.0)  # Default to 25 if unknown

    # ============ COMPUTE CURRENT MALL METRICS ============
    mall_perf = current_store_performance.loc[
        current_store_performance.index.isin(mall_store_codes)
    ]
    mall_sri = current_store_sri.loc[current_store_sri.index.isin(mall_store_codes)]

    current_avg_sales = mall_perf[col.TARGET_SALES_PER_SQM].mean()
    current_avg_dwell = mall_perf[col.TARGET_DWELL_TIME].mean()
    # GLA-weighted SRI
    current_avg_sri = compute_gla_weighted_sri(mall_sri, mall_store_gla)

    # ============ COMPUTE NEW MALL METRICS (after swap) ============
    # Update sales
    new_mall_sales = mall_perf[col.TARGET_SALES_PER_SQM].copy()
    new_mall_sales.loc[store_to_swap] = pred_sales
    new_avg_sales = new_mall_sales.mean()

    # Update dwell
    new_mall_dwell = mall_perf[col.TARGET_DWELL_TIME].copy()
    new_mall_dwell.loc[store_to_swap] = pred_dwell
    new_avg_dwell = new_mall_dwell.mean()

    # Update SRI (use category average for new store) - GLA-weighted
    new_mall_sri = mall_sri.copy()
    if store_to_swap in new_mall_sri.index:
        new_mall_sri.loc[store_to_swap] = pred_sri
    else:
        new_mall_sri = pd.concat(
            [new_mall_sri, pd.Series([pred_sri], index=[store_to_swap])]
        )
    new_avg_sri = compute_gla_weighted_sri(new_mall_sri, mall_store_gla)

    # ============ COMPUTE COMPOSITE SCORES ============
    current_composite = compute_mall_composite_score(
        current_avg_sales, current_avg_dwell, current_avg_sri, weights
    )
    new_composite = compute_mall_composite_score(
        new_avg_sales, new_avg_dwell, new_avg_sri, weights
    )

    composite_improvement = (
        (new_composite - current_composite) / current_composite
    ) * 100

    # ============ BUILD RESULT ============
    result = {
        "swapped_store": {
            "store_code": store_to_swap,
            "old_category": old_category,
            "new_category": new_category,
            "gla": store_gla,
            "gla_share": store_gla / mall_total_gla,
        },
        "current_mall_metrics": {
            "mall_id": mall_id,
            "avg_sales_per_sqm": current_avg_sales,
            "avg_dwell_time": current_avg_dwell,
            "avg_sri_gla_weighted": current_avg_sri,
            "composite_score": current_composite,
        },
        "predicted_mall_metrics": {
            "mall_id": mall_id,
            "avg_sales_per_sqm": new_avg_sales,
            "avg_dwell_time": new_avg_dwell,
            "avg_sri_gla_weighted": new_avg_sri,
            "composite_score": new_composite,
        },
        "improvement": {
            "sales_pct": ((new_avg_sales - current_avg_sales) / current_avg_sales)
            * 100,
            "dwell_pct": ((new_avg_dwell - current_avg_dwell) / current_avg_dwell)
            * 100,
            "sri_pct": ((new_avg_sri - current_avg_sri) / current_avg_sri) * 100,
            "composite_pct": composite_improvement,
        },
        "new_store_predictions": {
            "sales_per_sqm": pred_sales,
            "dwell_time": pred_dwell,
            "sri": pred_sri,
        },
    }

    return result

In [ ]:
# Create performance DataFrame indexed by store_code
current_performance = y.copy()
current_performance.index = store_codes
current_performance.index.name = col.STORE_CODE

# Create SRI Series indexed by store_code
current_store_sri = fact_sri_scores.set_index(col.STORE_CODE)[col.SRI_SCORE]

# Compute category-level SRI averages (for new stores)
category_sri_avg = compute_category_sri_averages(fact_sri_scores, dim_blocks)

print(f"Current performance shape: {current_performance.shape}")
print(f"Stores with SRI data: {len(current_store_sri)}")
print("\nCategory SRI averages:")
for cat, sri in sorted(category_sri_avg.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {sri:.1f}")

In [ ]:
# Example: Swap store 42 from "Food & Beverage Services" to "Fashion apparel"
# First, let's see what categories are available
print("Available categories:")
print(dim_blocks[col.CAT_HIGH].unique())

In [ ]:
store_total

In [ ]:
# Run the swap simulation with composite scoring
impact = predict_swap_impact(
    store_to_swap=186,
    new_category="Fitness",
    models=models,
    encoders=encoders,
    dim_blocks=dim_blocks,
    store_total=store_total,
    cross_visits=cross_visits,
    affinity_matrix=affinity,
    fact_sri_scores=fact_sri_scores,
    current_store_performance=current_performance,
    current_store_sri=current_store_sri,
    category_sri_avg=category_sri_avg,
    weights={"sales": 0.4, "dwell": 0.3, "sri": 0.3},  # Customize weights here
)

# Display results
print("=" * 70)
print("SWAP SIMULATION RESULTS")
print("=" * 70)

ss = impact["swapped_store"]
print(f"\nSWAP: Store {ss['store_code']}")
print(f"   {ss['old_category']} → {ss['new_category']}")
print(f"   GLA: {ss['gla']:.0f} sqm ({ss['gla_share'] * 100:.1f}% of mall)")

print(f"\nCURRENT MALL METRICS (Mall {impact['current_mall_metrics']['mall_id']}):")
cm = impact["current_mall_metrics"]
print(f"   Avg Sales/sqm:       {cm['avg_sales_per_sqm']:.3f}")
print(f"   Avg Dwell Time:      {cm['avg_dwell_time']:.3f}")
print(f"   Avg SRI (GLA-wtd):   {cm['avg_sri_gla_weighted']:.1f}")
print(f"   Composite Score:     {cm['composite_score']:.3f}")

print("\nPREDICTED MALL METRICS (after swap):")
pm = impact["predicted_mall_metrics"]
print(f"   Avg Sales/sqm:       {pm['avg_sales_per_sqm']:.3f}")
print(f"   Avg Dwell Time:      {pm['avg_dwell_time']:.3f}")
print(f"   Avg SRI (GLA-wtd):   {pm['avg_sri_gla_weighted']:.1f}")
print(f"   Composite Score:     {pm['composite_score']:.3f}")

print("\nIMPROVEMENT:")
imp = impact["improvement"]
print(f"   Sales:     {imp['sales_pct']:+.2f}%")
print(f"   Dwell:     {imp['dwell_pct']:+.2f}%")
print(f"   SRI:       {imp['sri_pct']:+.2f}%")
print(f"   COMPOSITE: {imp['composite_pct']:+.2f}%")

print("\nNEW STORE PREDICTIONS:")
nsp = impact["new_store_predictions"]
print(f"   Sales/sqm:  {nsp['sales_per_sqm']:.3f} (relative to mall avg)")
print(f"   Dwell Time: {nsp['dwell_time']:.3f} (relative to mall avg)")
print(f"   SRI:        {nsp['sri']:.1f} (category average)")